<a href="https://colab.research.google.com/github/aureamc/A04-clustering/blob/main/AI_Version_notebook_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Assignment: $k$ Means Clustering


**Q1.** This is a question about clustering. We want to investigate how adjusting the "noisiness" of the data impacts the quality of the algorithm and the difficulty of picking $k$.

1. Run the code below, which creates four datasets: `df0_125`, `df0_25`, `df0_5`, `df1_0`, and `df2_0`. Each data set is created by increasing the amount of `noise` (standard deviation) around the cluster centers, from `0.125` to `0.25` to `0.5` to `1.0` to `2.0`.

In [ ]:
import numpy as np
import pandas as pd

def createData(noise,N=50):
    np.random.seed(100) # Set the seed for replicability
    # Generate (x1,x2,g) triples:
    X1 = np.array([np.random.normal(1,noise,N),np.random.normal(1,noise,N)])
    X2 = np.array([np.random.normal(3,noise,N),np.random.normal(2,noise,N)])
    X3 = np.array([np.random.normal(5,noise,N),np.random.normal(3,noise,N)])
    # Concatenate into one data frame
    gdf1 = pd.DataFrame({'x1':X1[0,:],'x2':X1[1,:],'group':'a'})
    gdf2 = pd.DataFrame({'x1':X2[0,:],'x2':X2[1,:],'group':'b'})
    gdf3 = pd.DataFrame({'x1':X3[0,:],'x2':X3[1,:],'group':'c'})
    df = pd.concat([gdf1,gdf2,gdf3],axis=0)
    return df

df0_125 = createData(0.125)
df0_25 = createData(0.25)
df0_5 = createData(0.5)
df1_0 = createData(1.0)
df2_0 = createData(2.0)

2. Make scatterplots of the $(X1,X2)$ points by group for each of the datasets. As the `noise` goes up from 0.125 to 2.0, what happens to the visual distinctness of the clusters?
3. Create a scree plot for each of the datasets. Describe how the level of `noise` affects the scree plot (particularly the presence of a clear "elbow") and your ability to definitively select a $k$. (Pay attention to the vertical axis across plots, or put all the scree curves on a single canvas.)
4. Explain the intuition of the elbow, using this numerical simulation as an example.

**Q2.** This question is a case study on clustering.

1. Load the `SIPRI Military Expenditure Database.csv` file in the `./data` folder. This has data about military spending by country. Filter the rows to select only the year 2020, and drop all rows with missing values. I ended up with 148 countries. Is any further cleaning of the variables required?
2. Max-min normalize `Spending (2020 USD)` and `Spending per Capita`. Use a scree plot to determine the optimal number of clusters for the $k$ means clustering algorithm. Make a scatter plot of `Spending (2020 USD)` and `Spending per Capita`, and hue the dots by their cluster membership. Compute a describe table conditional on cluster membership (i.e. `.groupby(cluster).describe()`). What do you see? Where is the United States? Do you notice any patterns in the cluster membership?
3. Repeat part 2 for `Percent of Government Spending` and `Percent of GDP`. How do your results compare to part 2?
4. Use $k$ means clustering with all four numeric variables: `Spending (2020 USD)`, `Spending per Capita`, `Percent of Government Spending`, and `Percent of GDP`. How do your results compare to the previous two parts?
5. Did the $k$-MC algorithm find any useful patterns for you in analyzing the spending?

---

## *Phase 2: AI-assisted Revision
Keep Phase 1 frozen.

- **Phase 1 commit SHA ID:** [To be provided]
- **AI tool and model used:** Gemini

### *Prompts used

1.

Review my complete Python notebook for a k-Means clustering assignment. The assignment covers two parts. Q1 investigates data noisiness impact on cluster distinctness and scree plots across multiple generated datasets (df0_125 to df2_0), and Q2 is a case study using the SIPRI Military Expenditure dataset analyzing various subsets of spending variables.

2.

Can you help ensure the elbow method implementation for finding optimal k uses a clean loop over a range of cluster values? Then please check my code for vectorization efficiency, clean pandas/numpy syntax, correct min-max normalization, proper scree plot / elbow identification logic, and interpretation steps for the cluster descriptive statistics.

### *AI-proposed changes


1. Inertia Tracking for Scree Plots: Streamline the loop used to compute within-cluster sum of squares (inertia) across a range of k values to make the scree plot generation cleaner and reusable via a helper function.

2. Min-Max Normalization Vectorization: Replace manual min-max scaling arithmetic with scikit-learn's MinMaxScaler or explicit, robust pandas vectorization to avoid division-by-zero edge cases.

3. Data Cleaning & Missing Value Handling: Explicitly verify drop operations and data types on the SIPRI dataset before running clustering models to ensure no string columns or trailing whitespace cause type errors.

4. Visual Layout and Subplots: Structure Q1 scatterplots and scree plots using a clean grid layout (plt.subplots) so trends from low to high noise are easily readable side-by-side.

### *Evaluation of AI suggestions


*Evaluation of AI suggestions

Change	Decision	Reason and verification

1.	Accept | Using a standard loop to extract kmeans.inertia_ is clean and standard for building scree plots. Verified via scikit-learn documentation.

2.	Accept | Explicit manual scaling or MinMaxScaler ensures variables like total spending and per-capita spending are weighted evenly. Verified.

3.	Accept | Dropping missing values properly ensures the exact dataset size aligns with expectations. Verified.

4.	Accept | Grid subplots make comparing the visual impact of noise levels significantly clearer. Verified.

### *AI-assisted solution in full
After the evaluation of AI suggestions, incorporate the accepted revisions into your Phase 1 solution and provide the final Phase 2 solution in full in the following cells.

In [ ]:
# Your Phase 2 solution to the problem goes here.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler

# ==========================================
# Q1: Investigating Noisiness in k-Means
# ==========================================

# 1. Data Generation Function (Given)
def createData(noise, N=50):
    np.random.seed(100) # Set the seed for replicability
    X1 = np.array([np.random.normal(1, noise, N), np.random.normal(1, noise, N)])
    X2 = np.array([np.random.normal(3, noise, N), np.random.normal(2, noise, N)])
    X3 = np.array([np.random.normal(5, noise, N), np.random.normal(3, noise, N)])

    gdf1 = pd.DataFrame({'x1': X1[0, :], 'x2': X1[1, :], 'group': 'a'})
    gdf2 = pd.DataFrame({'x1': X2[0, :], 'x2': X2[1, :], 'group': 'b'})
    gdf3 = pd.DataFrame({'x1': X3[0, :], 'x2': X3[1, :], 'group': 'c'})
    df = pd.concat([gdf1, gdf2, gdf3], axis=0).reset_index(drop=True)
    return df

df0_125 = createData(0.125)
df0_25  = createData(0.25)
df0_5   = createData(0.5)
df1_0   = createData(1.0)
df2_0   = createData(2.0)

datasets = {
    'Noise: 0.125': df0_125,
    'Noise: 0.25': df0_25,
    'Noise: 0.5': df0_5,
    'Noise: 1.0': df1_0,
    'Noise: 2.0': df2_0
}

# 2. Scatterplots by Group
fig, axes = plt.subplots(1, 5, figsize=(20, 4), sharex=True, sharey=True)
for ax, (title, df) in zip(axes, datasets.items()):
    sns.scatterplot(data=df, x='x1', y='x2', hue='group', palette='Set1', ax=ax, alpha=0.8, legend=False)
    ax.set_title(title)
plt.suptitle("Scatterplots across Noise Levels", y=1.05)
plt.tight_layout()
plt.show()

# 3. Scree Plots for Each Dataset
def get_inertia(df, max_k=10):
    X = df[['x1', 'x2']].values
    inertias = []
    for k in range(1, max_k + 1):
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        kmeans.fit(X)
        inertias.append(kmeans.inertia_)
    return inertias

plt.figure(figsize=(8, 6))
for title, df in datasets.items():
    inertias = get_inertia(df)
    plt.plot(range(1, 11), inertias, marker='o', label=title)

plt.xlabel('Number of Clusters ($k$)')
plt.ylabel('Inertia (Within-Cluster Sum of Squares)')
plt.title('Scree Plots across Different Noise Levels')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

# ==========================================
# Q2: SIPRI Military Expenditure Case Study
# ==========================================

# 1. Load and Clean Data
# Assuming file is located in './data/SIPRI Military Expenditure Database.csv'
sipri_df = pd.read_csv('./data/SIPRI Military Expenditure Database.csv')

# Filter for the year 2020 and drop missing values across relevant columns
# Note: Column names might require minor adjustments based on exact CSV headers
sipri_2020 = sipri_df[sipri_df['Year'] == 2020].dropna()

# 2. Max-Min Normalization & k-Means on Absolute/Per Capita Spending
scaler = MinMaxScaler()
vars_part2 = ['Spending (2020 USD)', 'Spending per Capita']
scaled_data_2 = scaler.fit_transform(sipri_2020[vars_part2])

# Find optimal k using scree plot logic
inertias_q2 = []
for k in range(1, 11):
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(scaled_data_2)
    inertias_q2.append(km.inertia_)

# Fit optimal k (e.g., k=3 based on elbow)
optimal_k_2 = 3
kmeans_2 = KMeans(n_clusters=optimal_k_2, random_state=42, n_init=10)
sipri_2020['cluster_p2'] = kmeans_2.fit_predict(scaled_data_2)

# Scatter plot
plt.figure(figsize=(8, 6))
sns.scatterplot(data=sipri_2020, x='Spending (2020 USD)', y='Spending per Capita', hue='cluster_p2', palette='viridis')
plt.title('Military Spending vs Spending per Capita by Cluster')
plt.show()

# Describe table conditional on cluster membership
print(sipri_2020.groupby('cluster_p2')[vars_part2].describe())

# 3. Repeat for Percent of Government Spending and Percent of GDP
vars_part3 = ['Percent of Government Spending', 'Percent of GDP']
scaled_data_3 = scaler.fit_transform(sipri_2020[vars_part3])

kmeans_3 = KMeans(n_clusters=3, random_state=42, n_init=10)
sipri_2020['cluster_p3'] = kmeans_3.fit_predict(scaled_data_3)

print(sipri_2020.groupby('cluster_p3')[vars_part3].describe())

# 4. k-Means using All Four Numeric Variables
vars_all = ['Spending (2020 USD)', 'Spending per Capita', 'Percent of Government Spending', 'Percent of GDP']
scaled_data_all = scaler.fit_transform(sipri_2020[vars_all])

kmeans_all = KMeans(n_clusters=3, random_state=42, n_init=10)
sipri_2020['cluster_all'] = kmeans_all.fit_predict(scaled_data_all)

print(sipri_2020.groupby('cluster_all')[vars_all].describe())

### *Reflection on AI assistance
In your own words without generative AI.

The cooperation with AI on Phase 2 greatly helped with improving the overall structure and execution of the data generation simulation and the military expenditure case study. First of all, these optimizations were related to better code structure and performance, specifically with the introduction of helper functions for calculating cluster inertia and looping through plots. This allowed making it easy to observe how an increase in noise deteriorates the visibility of the elbow on the scree plot. In case of the SIPRI case study, applying MinMaxScaler helped in balancing the values of the features which have dramatically different magnitudes in order to not to let the absolute spending overshadow the relative percentage-based ones.

Nonetheless, in some cases, the AI assumed generic column names which should be corrected manually in order to make sure that they perfectly match the original source data. The verification process was done by checking the clustering centers and confirming that the large global outlier like the United States correctly appeared in the cluster in case of absolute spending and was mixed with similar countries in the case of percentage-spending evaluation.


